In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
base_dir = Path(
    "/Volumes/shared/pyne_group/Shared/AFM_Data/Plasmids/pICoZ/20260105_20251031_20251107_combined_picoz_dataset/202608XX-response-to-reviewers"
)
assert base_dir.exists()

child_directories = [directory for directory in base_dir.iterdir() if directory.is_dir()]
print(f"Found {len(child_directories)} child directories in {base_dir.name}:")
for directory in child_directories:
    print(f" - {directory.name}")

datasets = {}
for child_directory in child_directories:
    # see if there is a file called "grain_statistics.csv"
    grainstats_file = child_directory / "grain_statistics.csv"
    if grainstats_file.exists():
        print(f"Found grain_statistics.csv in {child_directory.name}")
        datasets[child_directory.name] = {
            "grain_stats": pd.read_csv(grainstats_file),
        }
    else:
        print(f"No grain_statistics.csv found in {child_directory.name}")

# plot stats between them

In [ ]:
column_to_plot = "total_contour_length"
column_scaling_factor = 1e9
expected_value = 434
for dataset_name, dataset in datasets.items():
    grain_stats = dataset["grain_stats"]
    if column_to_plot in grain_stats.columns:
        print(f"Dataset: {dataset_name}, {column_to_plot} stats:")
        print(grain_stats[column_to_plot].describe())
    else:
        print(f"Dataset: {dataset_name} does not have column {column_to_plot}")
        print(f"Columns available: {grain_stats.columns.tolist()}")

# plot a stripplot of the stat for each dataset
fig, ax = plt.subplots(figsize=(10, 6))
for dataset_name, dataset in datasets.items():
    grain_stats = dataset["grain_stats"]
    if column_to_plot in grain_stats.columns:
        sns.stripplot(
            x=[dataset_name] * len(grain_stats),
            y=grain_stats[column_to_plot] * column_scaling_factor,
            ax=ax,
            jitter=True,
            alpha=0.5,
        )
        # draw a boxplot on top of the stripplot
        sns.boxplot(
            x=[dataset_name] * len(grain_stats),
            y=grain_stats[column_to_plot] * column_scaling_factor,
            ax=ax,
            showcaps=True,
            boxprops={"facecolor": "None"},
            showfliers=False,
            whiskerprops={"linewidth": 2},
        )
# add a horizontal line for the expected value
ax.axhline(expected_value, color="red", linestyle="--", label=f"Expected value: {expected_value}")
ax.set_title(f"Comparison of {column_to_plot} across datasets")
ax.set_ylabel(column_to_plot)
# rotate x-axis labels for better readability
plt.xticks(rotation=45)
ax.legend()
plt.show()